In [1]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate
from evidently.ui.workspace import CloudWorkspace

## Configuration credential

In [2]:
# Configuration for the credential
import os
from dotenv import load_dotenv

ENV_PATH = ".env"
load_dotenv(ENV_PATH)

False

## Create project on evidently AI

In [3]:
ws = CloudWorkspace(token=os.getenv("EVIDENTLY_TOKEN"), url="https://app.evidently.cloud")

## Create a Project within your Organization, or connect to an existing Project:

In [4]:
try:
    existing_projects = ws.list_projects()
    project = next((p for p in existing_projects if p.name == "LLM Audit"), None)

    if project is None:
        # Create a new project if it doesn't exist
        project = ws.create_project(name="LLM Audit", org_id="019d6885-06c6-7601-9a3e-2298bed991f4")
        project.description = "Project for auditing LLM performance on models, data, and LLMs evaluations."
        project.save()
        project(f"✅ Project created: {project.name}")
    else:
        project(f"📁 Project already exists: {project.name}")
except Exception as e:
    print(f"Error accessing or creating project: {e}")

Error accessing or creating project: 'Project' object is not callable


In [5]:
# See the project exists
print(f"Project name: {project.name}")

Project name: LLM Audit


## Prepare the dataset

In [6]:
data = [
        ["What is the fraud detection rate in the dataset?", "Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.",
        "The dataset shows a fraud label distribution. High precision minimises false positives."],
        ["Which ML model performs best for credit default prediction?",
        "XGBoost and Random Forest outperform logistic regression on imbalanced datasets.",
        "XGBoost typically achieves the highest ROC-AUC for credit default tasks."],
        ["How should missing values be handled in operational banking data?",
        "Imputation must preserve regulatory compliance and not introduce synthetic bias.",
        "Use median for numerical and mode for categorical. Flag missingness as a binary indicator."],
        ["What is the churn rate and how can it be reduced?",
        "Customer churn in retail banking averages 15-20% annually.",
        "Churn column shows the distribution. Targeted retention can reduce churn by 10-15%."],
        ["Explain the class imbalance issue in fraud detection.",
        "Fraud datasets are heavily imbalanced with fraud rates below 1-2%.",
        "Requires resampling or class-weight adjustment. Use F1 and PR-AUC, not accuracy."],
]
columns = ["question", "context", "answer"]

pd.set_option('display.max_colwidth', None)
eval_df = pd.DataFrame(data, columns=columns)
eval_df

,question,context,answer
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,The dataset shows a fraud label distribution. High precision minimises false positives.
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,XGBoost typically achieves the highest ROC-AUC for credit default tasks.
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.,Use median for numerical and mode for categorical. Flag missingness as a binary indicator.
3,What is the churn rate and how can it be reduced?,Customer churn in retail banking averages 15-20% annually.,Churn column shows the distribution. Targeted retention can reduce churn by 10-15%.
4,Explain the class imbalance issue in fraud detection.,Fraud datasets are heavily imbalanced with fraud rates below 1-2%.,"Requires resampling or class-weight adjustment. Use F1 and PR-AUC, not accuracy."


## Auditing the LLMs for evaluation

* ###  Sentiment: from -1 (negative) to 1 (positive)
* ### Text length: character count
* ### Denials: refusals to answer. This uses an LLM-as-a-judge with built-in prompt.

### Each evaluation is a descriptor. It adds a new score or label to each row in your dataset. For LLM-as-a-judge, we’ll use OpenAI GPT-4o mini. Set OpenAI key as an environment variable:

In [7]:
import os
from groq import Groq

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY_2"))

def get_groq_response(prompt):
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1024,
        top_p=0.9,
        stream=False
    )
    return completion.choices[0].message.content

# Generate answers
eval_df['answer'] = eval_df['question'].apply(get_groq_response)
eval_df

,question,context,answer
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"This conversation has just begun. I'm happy to chat with you, but I don't have any information about a dataset or its fraud detection rate. If you'd like to share more context or provide a dataset, I'd be happy to try and help you calculate the fraud detection rate."
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"The best machine learning (ML) model for credit default prediction can vary depending on the specific dataset, features, and performance metrics used. However, based on various studies and benchmarks, here are some of the top-performing ML models for credit default prediction:\n\n1. **Random Forest**: Random Forest is a popular and widely used ensemble method that has been shown to perform well in credit default prediction tasks. It can handle high-dimensional data and is robust to outliers and missing values.\n2. **Gradient Boosting**: Gradient Boosting is another ensemble method that has been shown to be effective in credit default prediction. It can handle complex interactions between variables and is robust to overfitting.\n3. **Neural Networks**: Neural Networks, particularly those with multiple layers, have been shown to be effective in credit default prediction. They can learn complex patterns in the data and are particularly useful when dealing with large datasets.\n4. **Support Vector Machines (SVMs)**: SVMs are a type of supervised learning algorithm that can be used for credit default prediction. They are particularly useful when dealing with high-dimensional data and can handle non-linear relationships between variables.\n5. **XGBoost**: XGBoost is a type of Gradient Boosting algorithm that has been shown to be highly effective in credit default prediction tasks. It is particularly useful when dealing with large datasets and can handle complex interactions between variables.\n\nSome of the key factors to consider when selecting an ML model for credit default prediction include:\n\n* **Data quality and availability**: The quality and availability of the data can have a significant impact on the performance of the ML model.\n* **Feature engineering**: The selection and engineering of relevant features can have a significant impact on the performance of the ML model.\n* **Model complexity**: The complexity of the ML model can have a significant impact on its performance, with more complex models requiring larger datasets and more computational resources.\n* **Hyperparameter tuning**: The hyperparameters of the ML model can have a significant impact on its performance, and tuning them can be time-consuming and require significant expertise.\n\nSome of the key performance metrics used to evaluate the performance of ML models for credit default prediction include:\n\n* **Accuracy**: The proportion of correctly classified instances.\n* **Precision**: The proportion of true positives among all positive predictions.\n* **Recall**: The proportion of true positives among all actual positive instances.\n* **F1-score**: The harmonic mean of precision and recall.\n* **Area under the ROC curve (AUC-ROC)**: A measure of the model's ability to distinguish between positive and negative classes.\n* **Area under the precision-recall curve (AUC-PR)**: A measure of the model's ability to distinguish between positive and negative classes, with a focus on precision and recall.\n\nSome of the key datasets used for credit default prediction include:\n\n* **German Credit Dataset**: A widely used dataset that contains information on credit applicants, including demographic and financial information.\n* **Australian Credit Approval Dataset**: A dataset that contains information on credit applicants, including demographic and financial information.\n* **Lending Club Dataset**: 

In [ ]:
import litellm
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="litellm")

# litellm specifically looks for this key name
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY_2", "")

# Verify the key is actually loaded (should NOT be empty)
print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

# Optional: turn on debug to see full LiteLLM logs
litellm._turn_on_debug()

eval_dataset = Dataset.from_pandas(
    eval_df,
    data_definition=DataDefinition(),
    descriptors=[
        Sentiment("answer", alias="Sentiment"),
        TextLength("answer", alias="Length"),
        DeclineLLMEval(
            "answer",
            alias="Denials",
            provider="litellm",
            model="groq/llama-3.3-70b-versatile"
        ),
    ]
)

eval_dataset.as_dataframe()

15:32:35 - LiteLLM:DEBUG: utils.py:481 - 

15:32:35 - LiteLLM:DEBUG: utils.py:481 - Request to litellm:
15:32:35 - LiteLLM:DEBUG: utils.py:481 - litellm.acompletion(model='groq/llama-3.3-70b-versatile', custom_llm_provider=None, messages=[{'role': 'system', 'content': 'You are a judge which evaluates text.'}, {'role': 'user', 'content': 'A "DECLINE" typically refers to a refusal or a polite rejection to do something.\nIn these contexts, "DECLINE" signifies a respectful or formal way of saying no to provide a help, service, or answer.\nClassify text between ___text_starts_here___ and ___text_ends_here___ into two categories: DECLINE and OK.\n___text_starts_here___\nThis conversation has just begun. I\'m happy to chat with you, but I don\'t have any information about a dataset or its fraud detection rate. If you\'d like to share more context or provide a dataset, I\'d be happy to try and help you calculate the fraud detection rate.\n___text_ends_here___\nUse the following categories for 

Key loaded: True


15:32:35 - LiteLLM:DEBUG: litellm_logging.py:1223 - RAW RESPONSE:
{"id": "chatcmpl-6384e8b4-03fa-45a7-9905-013ff22a42d1", "object": "chat.completion", "created": 1775719955, "model": "llama-3.3-70b-versatile", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\"category\": \"OK\", \"reasoning\": \"The text does not contain a refusal or rejection, but instead an offer to help and a request for more information, indicating a willingness to assist.\"}"}, "logprobs": null, "finish_reason": "stop"}], "usage": {"queue_time": 0.045798345, "prompt_tokens": 262, "prompt_time": 0.016698495, "completion_tokens": 42, "completion_time": 0.172205709, "total_tokens": 304, "total_time": 0.188904204}, "usage_breakdown": null, "system_fingerprint": "fp_dae98b5ecb", "x_groq": {"id": "req_01knrjgf1ze7nswqee5gww3hhk", "seed": 1514989707}, "service_tier": "on_demand"}


15:32:35 - LiteLLM:DEBUG: litellm_logging.py:3292 - Filtered callbacks: []
15:32:35 - LiteLLM:DEBUG: cost_calculator.p

,question,context,answer,Sentiment,Length,Denials,Denials reasoning
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,"This conversation has just begun. I'm happy to chat with you, but I don't have any information about a dataset or its fraud detection rate. If you'd like to share more context or provide a dataset, I'd be happy to try and help you calculate the fraud detection rate.",0.6808,266,OK,"The text does not contain a refusal or rejection, but instead an offer to help and a request for more information, indicating a willingness to assist."
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,"The best machine learning (ML) model for credit default prediction can vary depending on the specific dataset, features, and performance metrics used. However, based on various studies and benchmarks, here are some of the top-performing ML models for credit default prediction:\n\n1. **Random Forest**: Random Forest is a popular and widely used ensemble method that has been shown to perform well in credit default prediction tasks. It can handle high-dimensional data and is robust to outliers and missing values.\n2. **Gradient Boosting**: Gradient Boosting is another ensemble method that has been shown to be effective in credit default prediction. It can handle complex interactions between variables and is robust to overfitting.\n3. **Neural Networks**: Neural Networks, particularly those with multiple layers, have been shown to be effective in credit default prediction. They can learn complex patterns in the data and are particularly useful when dealing with large datasets.\n4. **Support Vector Machines (SVMs)**: SVMs are a type of supervised learning algorithm that can be used for credit default prediction. They are particularly useful when dealing with high-dimensional data and can handle non-linear relationships between variables.\n5. **XGBoost**: XGBoost is a type of Gradient Boosting algorithm that has been shown to be highly effective in credit default prediction tasks. It is particularly useful when dealing with large datasets and can handle complex interactions between variables.\n\nSome of the key factors to consider when selecting an ML model for credit default prediction include:\n\n* **Data quality and availability**: The quality and availability of the data can have a significant impact on the performance of the ML model.\n* **Feature engineering**: The selection and engineering of relevant features can have a significant impact on the performance of the ML model.\n* **Model complexity**: The complexity of the ML model can have a significant impact on its performance, with more complex models requiring larger datasets and more computational resources.\n* **Hyperparameter tuning**: The hyperparameters of the ML model can have a significant impact on its performance, and tuning them can be time-consuming and require significant expertise.\n\nSome of the key performance metrics used to evaluate the performance of ML models for credit default prediction include:\n\n* **Accuracy**: The proportion of correctly classified instances.\n* **Precision**: The proportion of true positives among all positive predictions.\n* **Recall**: The proportion of true positives among all actual positive instances.\n* **F1-score**: The harmonic mean of precision and recall.\n* **Area under the ROC curve (AUC-ROC)**: A measure of the model's ability to distinguish between positive and negative classes.\n* **Area under the precision-recall curve (AUC-PR)**: A measure of the model's ability to distinguish between positive and negative classes, with a focus on precision and recall.\n\nSome of the key datasets used for credit default prediction include:\n\n* **German Credit Dataset**: A widely used dataset that contains information on credit applicants, including demograp